# Environment audit

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet **ON** (notebooks 02–06 download the 8 pinned helper wheels, ~57 MB, and 05/06 download the model). Run notebooks in order 01 → 06 (each Kaggle notebook is a fresh session, so notebooks 03–06 re-run the idempotent installer first); each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Prereq: release dataset attached. Success: every check prints PASS (Python 3.13, torch 2.11.0+cu130, Triton 3.6, glibc>=2.38, 2x Tesla T4 SM75). Peer access/topology are recorded, not assumed.

## 1. Setup — locate the release bundle

In [ ]:
import glob, json, os, pathlib
# Explicit release identity: this notebook accepts ONLY the bundle directory named RELEASE whose RELEASE-MANIFEST.json says the same name.
# (Several Kaggle datasets may be attached; an older rc is never silently selected.) Override the search root with STOKASTISK_BUNDLE.
RELEASE = "stokastisk-0.3.0-kaggle-t4-sm75-rc5"
cands = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
if os.environ.get("STOKASTISK_BUNDLE"): cands = [pathlib.Path(os.environ["STOKASTISK_BUNDLE"])]
match = [c for c in cands if c.name == RELEASE]
assert match, f"release {RELEASE!r} not found. Attached bundles: {sorted(str(c) for c in cands)}. Attach the {RELEASE} dataset."
assert len({(c / "manifests" / "SHA256SUMS").read_bytes() for c in match}) == 1, f"several DIFFERENT copies of {RELEASE}: {match}"
BUNDLE = match[0]
_m = json.loads((BUNDLE / "manifests" / "RELEASE-MANIFEST.json").read_text())
assert _m.get("name") == RELEASE, f"manifest name {_m.get('name')!r} != expected {RELEASE!r}"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("release:", RELEASE, "| bundle:", BUNDLE, "| other attached bundles ignored:", sorted(str(c) for c in cands if c != BUNDLE))

## 2. Run the audit (read-only: installs and changes nothing)

In [ ]:
import subprocess, sys, os
def run_stage(script, *args):
    """Run a bundled stage script; returns its exit code (a failure is asserted in the final verdict cell, after evidence is packaged)."""
    r = subprocess.run([sys.executable, str(SCRIPTS / script), *args], cwd=SCRIPTS,
                       env=dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", "")))
    print(f"{script} exit code: {r.returncode}")
    return r.returncode

In [ ]:
RC = {"01_env_audit": run_stage("01_env_audit.py")}

## Package the evidence

Creates `/kaggle/working/01-stokastisk-acceptance.zip` (download it from the Output tab).

In [ ]:
import shutil
print(shutil.make_archive("/kaggle/working/01-stokastisk-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))

## Verdict — this cell fails the notebook if any stage failed

In [ ]:
rcs = RC
assert all(v == 0 for v in rcs.values()), f'FAILED stage(s): {rcs} -- see the logs and JSON in /kaggle/working/stokastisk-acceptance/'
print('ALL STAGES PASSED', rcs)